# Cost-Aware FinQA — GRPO Training for Cost-Efficient Tool SelectionThis notebook trains a small LLM (Qwen 2.5-1.5B) using **GRPO (Group Relative Policy Optimization)**with **Unsloth** (4-bit LoRA) to learn cost-aware tool selection for financial question answering.**Principle:** The model should learn to discover the most cost-effective path to a solutionwithout affecting its overall knowledge and general-purpose ability.**Phases:**1. Install dependencies2. Clone environment & verify HF Space3. Import local environment & sanity check4. Load model with Unsloth (4-bit LoRA)5. Prompt engineering & action parsing6. Reward shaping (dense signal for GRPO)7. Episode runner8. Random agent baseline9. Pre-training LLM baseline10. **GRPO Training** (primary methodology)11. LLM-as-Judge evaluation12. Post-training evaluation13. Comparison plots14. Detailed episode walkthrough15. Final summary

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q unsloth trl datasets transformers accelerate
!pip install -q pydantic requests matplotlib numpy
!pip install -q openenv python-dotenv

## Cell 2 — Clone Environment from GitHub & Verify HF Space

In [ ]:
import os, sys, json, requests

# ── HF_TOKEN setup for Colab ──────────────────────────────────────────────
# Option 1: Set via Colab secrets (recommended)
# Option 2: Set manually below
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        # Uncomment and set your token if not using Colab secrets:
        # os.environ["HF_TOKEN"] = "hf_your_token_here"
        print("WARNING: HF_TOKEN not set. Set it in Colab secrets or uncomment the line above.")
    else:
        print("HF_TOKEN found in environment")# ── Clone the environment code from GitHub ──────────────────────────────────REPO_DIR = "/content/cost_aware_finqa"if not os.path.exists(REPO_DIR):    os.system(f"git clone https://github.com/nsharan2000/cost-aware-finqa.git {REPO_DIR}")    print(f"Cloned environment code to {REPO_DIR}")else:    os.system(f"cd {REPO_DIR} && git pull")    print(f"Updated code at {REPO_DIR}")# Add to path so we can import the environment locallysys.path.insert(0, REPO_DIR)# ── Verify the deployed HF Space is live ───────────────────────────────────SPACE_URL = "https://Teachafy-cost-aware-finqa.hf.space"try:    resp = requests.post(f"{SPACE_URL}/reset", json={}, timeout=15)    obs = resp.json().get("observation", {})    print(f"HF Space is live at {SPACE_URL}")    print(f"  Question: {obs.get('question', '')[:80]}...")    print(f"  Budget: ${obs.get('budget_total', 0):.2f}")    print(f"  Tools: {obs.get('available_tools', [])}")except Exception as e:    print(f"Space not reachable ({e}). Training will use local env only.")

## Cell 3 — Import Local Environment & Sanity Check

In [ ]:
from server.cost_aware_finqa_environment import CostAwareFinqaEnvironment, TASK_CONFIGfrom models import CostAwareFinqaActionfrom server.tools import TOOL_COSTSprint(f"Environment imported successfully")print(f"Tasks: {list(TASK_CONFIG.keys())}")print(f"Tool costs: {TOOL_COSTS}")# Quick sanity checkenv = CostAwareFinqaEnvironment()obs = env.reset()print(f"\nLocal reset OK")print(f"  Question: {obs.question[:80]}...")print(f"  Budget: ${obs.budget_total:.2f}")print(f"  Max steps: {obs.max_steps}")print(f"  Available tools: {obs.available_tools}")print(f"  Table schema: {obs.table_schema[:120]}...")

## Cell 4 — Load Qwen 2.5-1.5B with Unsloth (4-bit, LoRA)

In [ ]:
from unsloth import FastLanguageModelimport torchMODEL_NAME  = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"MAX_SEQ_LEN = 2048LORA_R      = 16   # Low rank to preserve general-purpose abilitymodel, tokenizer = FastLanguageModel.from_pretrained(    model_name=MODEL_NAME,    max_seq_length=MAX_SEQ_LEN,    load_in_4bit=True,)model = FastLanguageModel.get_peft_model(    model,    r=LORA_R,    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",                     "gate_proj", "up_proj", "down_proj"],    lora_alpha=16,       # alpha = rank -> scaling factor = 1.0    lora_dropout=0,      # No dropout for stability    bias="none",    use_gradient_checkpointing="unsloth",)print(f"Model loaded: {MODEL_NAME}")model.print_trainable_parameters()

In [ ]:
# Enable fast inference mode (Unsloth optimization)FastLanguageModel.for_inference(model)

## Cell 5 — Prompt Engineering & Action Parsing

In [ ]:
import re, random, textwrapimport numpy as npfrom collections import defaultdict# ── Tool descriptions for the prompt ───────────────────────────────────────TOOL_DESCS = {    "sql_query":      "Run SQL on financial tables. $0.001 per call. Penalized for bad SQL (-0.15).",    "web_search":     "External web search. $0.02 per call. Only for industry benchmarks/comparisons.",    "upgrade_llm":    "Stronger model for reasoning. $1.00 per call. EXTREMELY EXPENSIVE (1000x SQL) — absolute last resort only.",    "submit_answer":  "Submit final answer. FREE. Answer should be a number or short phrase.",}SYSTEM_PROMPT = textwrap.dedent("""\You are a cost-aware financial research agent. You answer financial questions bychoosing the cheapest tool that gets the job done.Available tools:{tool_lines}Strategy:- Use sql_query first (cheapest at $0.001).- Use 'SELECT * FROM table_catalog' to discover available tables.- Only use expensive tools when sql_query can't answer the question.- Avoid redundant SQL calls — each one costs $0.001 and bad ones cost -0.15 penalty.- Minimize total cost to maximize your score.Scoring: correctness * (1 - cost/budget) * (1 - error_penalty)Numerical matching: <=1% error = 1.0, <=5% = 0.6, >5% = 0.0Respond with ONLY a JSON object:{{"tool": "<tool_name>", "query": "<your query>", "answer": "<if submitting>"}}""")def build_chat_messages(obs, last_result=None):    tool_lines = "\n".join(f"  - {name}: {desc}" for name, desc in TOOL_DESCS.items())    system = SYSTEM_PROMPT.format(tool_lines=tool_lines)    user_parts = [        f"Question: {obs.question}",        f"Budget: ${obs.budget_remaining:.2f} / ${obs.budget_total:.2f}",        f"Step: {obs.step_number} / {obs.max_steps}",    ]    if obs.table_schema:        user_parts.append(f"Table Schema:\n{obs.table_schema[:400]}")    if last_result:        user_parts.append(f"Last tool result:\n{last_result[:300]}")    if obs.error:        user_parts.append(f"Error: {obs.error}")    return [        {"role": "system", "content": system},        {"role": "user", "content": "\n\n".join(user_parts)},    ]def parse_action(text):    text = text.strip()    try:        if "{" in text:            start = text.index("{")            end = text.rindex("}") + 1            blob = json.loads(text[start:end])            return (                blob.get("tool", "submit_answer"),                blob.get("query", ""),                blob.get("answer", ""),            )    except (json.JSONDecodeError, ValueError):        pass    return ("submit_answer", "", text[:100])def generate_action(model, tokenizer, obs, last_result=None, temperature=0.7):    messages = build_chat_messages(obs, last_result)    prompt = tokenizer.apply_chat_template(        messages, tokenize=False, add_generation_prompt=True,    )    inputs = tokenizer(        prompt, return_tensors="pt",        truncation=True, max_length=MAX_SEQ_LEN,    ).to(model.device)    with torch.no_grad():        ids = model.generate(            **inputs, max_new_tokens=128,            temperature=temperature,            do_sample=True, top_p=0.95,        )    gen_text = tokenizer.decode(ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)    tool, query, answer = parse_action(gen_text)    return gen_text, {"tool": tool, "query": query, "answer": answer}print("Prompt engineering & action parsing ready.")

## Cell 6 — Reward Shaping (Dense Signal for GRPO)GRPO trains by comparing rewards across a group of completions for the same prompt.The reward function must provide a strong, dense signal that captures:- Format correctness (valid JSON)- Tool cost-awareness (prefer cheap tools)- SQL quality (penalize bad queries)- Process quality (not just final answer)

In [ ]:
# Per-tool step rewards (mirrors the environment's reward structure)TOOL_STEP_REWARDS = {    "sql_query":      0.03,   # Cheap and usually the right first step    "web_search":     0.02,   # Moderately priced, sometimes necessary for external data    "upgrade_llm":   -0.10,   # Extremely expensive ($1.00), absolute last resort    "submit_answer":  0.00,   # Neutral (final score matters)}def shaped_reward(gen_text, action, obs):    """Dense reward signal for GRPO training.    Components:      - Format bonus: +0.10 if valid JSON with tool field      - Tool choice: reward from TOOL_STEP_REWARDS      - Cost awareness: bonus for using cheap tools first      - SQL quality: penalty for errors, bonus for valid results      - Budget awareness: penalty for wasteful spending    """    reward = 0.0    # Format bonus — GRPO needs this to learn structured output    try:        blob = json.loads(gen_text.strip()) if "{" in gen_text else None        if blob and isinstance(blob, dict) and "tool" in blob:            reward += 0.10            if "query" in blob or "answer" in blob:                reward += 0.05    except (json.JSONDecodeError, ValueError):        reward -= 0.10  # Unparseable output    # Tool choice reward    tool = action.get("tool", "")    reward += TOOL_STEP_REWARDS.get(tool, 0.0)    # Cost awareness: bonus for using cheap tools early in episode    if tool == "sql_query" and obs.step_number <= 2:        reward += 0.05  # Good: exploring with cheap tool first    # Penalty for using expensive tools when budget is tight    tool_cost = TOOL_COSTS.get(tool, 0.0)    if tool_cost >= 1.0 and obs.budget_remaining < obs.budget_total * 0.5:        reward -= 0.05  # Wasteful spending    # Error penalty    if obs.error:        reward -= 0.10    return rewardprint("Reward shaping ready.")

## Cell 7 — Episode Runner

In [ ]:
def run_episode(    model,    tokenizer,    task: str = "basic_retrieval",    seed: int = None,    max_steps: int = 8,    temperature: float = 0.7,    verbose: bool = False,) -> dict:    """Run a full multi-step episode, return trajectory + rewards."""    ep_seed = seed or random.randint(0, 999_999)    os.environ["FINQA_TASK"] = task    env = CostAwareFinqaEnvironment()    random.seed(ep_seed)    env._question_index = ep_seed % max(len(env._task_questions) if env._task_questions else 1, 1)    obs = env.reset()    trajectory = []    total_env_reward = 0.0    total_shaped = 0.0    last_result = None    tools_used = set()    for step_i in range(max_steps):        gen_text, action = generate_action(            model, tokenizer, obs, last_result, temperature=temperature        )        act = CostAwareFinqaAction(            tool=action["tool"],            query=action["query"],            answer=action["answer"],        )        obs = env.step(act)        env_reward = obs.reward        total_env_reward += env_reward        last_result = obs.tool_result        tools_used.add(action["tool"])        sr = shaped_reward(gen_text, action, obs)        total_shaped += sr        trajectory.append({            "step": step_i,            "action": action,            "gen_text": gen_text[:200],            "env_reward": env_reward,            "shaped_rw": sr,            "tool_cost": obs.tool_cost,        })        if verbose:            cost_str = f"${obs.tool_cost:.2f}" if obs.tool_cost > 0 else "FREE"            print(f"  [{step_i:02d}] {action['tool']:<16s} ({cost_str})  "                  f"env={env_reward:+.4f}  shaped={sr:+.4f}  "                  f"gen='{gen_text[:55]}...'")        if obs.done:            break    return {        "trajectory":     trajectory,        "total_reward":   total_env_reward,        "total_shaped":   total_shaped,        "final_score":    obs.score,        "cost_spent":     obs.cost_so_far,        "budget_total":   obs.budget_total,        "steps":          len(trajectory),        "tools_used":     len(tools_used),        "tools":          tools_used,        "done":           obs.done,        "question":       obs.question[:80],        "task":           task,    }print("Episode runner ready.")

## Cell 8 — Phase 1: Random Agent Baseline

In [ ]:
print("=" * 60)print("Phase 1 — Random Agent Baseline")print("=" * 60)NUM_RANDOM = 10random_scores = []random_costs = []VALID_TOOLS = ["sql_query", "web_search", "upgrade_llm", "submit_answer"]for i in range(NUM_RANDOM):    os.environ["FINQA_TASK"] = random.choice(list(TASK_CONFIG.keys()))    env = CostAwareFinqaEnvironment()    obs = env.reset()    for _ in range(obs.max_steps):        tool = random.choice(VALID_TOOLS)        query = ""        answer = ""        if tool == "sql_query":            query = random.choice([                "SELECT * FROM table_catalog LIMIT 5",                "SELECT * FROM table_catalog",                "SELECT name FROM sqlite_master WHERE type='table'",            ])        elif tool == "web_search":            query = "company financial benchmarks"        elif tool == "submit_answer":            answer = str(random.uniform(0, 1))        act = CostAwareFinqaAction(tool=tool, query=query, answer=answer)        obs = env.step(act)        if obs.done:            break    random_scores.append(obs.score)    random_costs.append(obs.cost_so_far)    print(f"  Random ep {i+1:02d}: score={obs.score:.4f}  cost=${obs.cost_so_far:.2f}")print(f"  >> Random baseline: {np.mean(random_scores):.4f} +/- {np.std(random_scores):.4f}")print(f"  >> Avg cost: ${np.mean(random_costs):.2f}")

## Cell 9 — Phase 2: Pre-Training LLM Baseline

In [ ]:
print("\n" + "=" * 60)print("Phase 2 — Pre-Training LLM Baseline")print("=" * 60)NUM_EVAL = 8SEEDS = [i * 137 for i in range(NUM_EVAL)]EVAL_TASKS = ["basic_retrieval", "basic_retrieval", "analytical_reasoning",              "analytical_reasoning", "strategic_research", "strategic_research",              "basic_retrieval", "analytical_reasoning"]baseline_scores = []baseline_shaped = []baseline_costs = []baseline_steps = []for i, (seed, task) in enumerate(zip(SEEDS, EVAL_TASKS)):    res = run_episode(model, tokenizer, task=task, seed=seed,                      verbose=(i == 0))    baseline_scores.append(res["final_score"])    baseline_shaped.append(res["total_shaped"])    baseline_costs.append(res["cost_spent"])    baseline_steps.append(res["steps"])    print(f"  Ep {i+1:02d} [{task[:8]}]: score={res['final_score']:.4f}  "          f"shaped={res['total_shaped']:+.4f}  "          f"cost=${res['cost_spent']:.2f}  "          f"steps={res['steps']}  tools={res['tools']}")print(f"  >> Pre-train score mean:  {np.mean(baseline_scores):.4f} +/- {np.std(baseline_scores):.4f}")print(f"  >> Pre-train shaped mean: {np.mean(baseline_shaped):.4f} +/- {np.std(baseline_shaped):.4f}")print(f"  >> Pre-train avg cost:    ${np.mean(baseline_costs):.2f}")

## Cell 10 — Phase 3: GRPO Training**Group Relative Policy Optimization (GRPO)** generates multiple completions per prompt,computes rewards for each, and uses the relative ranking within the group to update the policy.Key safeguards against overtraining:- **Low LoRA rank (r=16)** — limits capacity for memorization- **KL penalty (beta=0.1)** — keeps model close to its pretrained distribution- **Early stopping** — monitors reward plateau- **Small number of training steps** — avoids catastrophic forgetting

In [ ]:
from datasets import Datasetfrom trl import GRPOTrainer, GRPOConfig# Ensure pad token is setif tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_tokentokenizer.padding_side = "right"print("\n" + "=" * 60)print("Phase 3 — GRPO Training")print("=" * 60)# ── Config ─────────────────────────────────────────────────────────────────NUM_PROMPTS     = 30    # Distinct episode promptsGROUP_SIZE      = 4     # Completions per prompt (GRPO group)NUM_EPOCHS      = 2     # Training epochsLEARNING_RATE   = 5e-5  # Conservative LR to prevent overtrainingKL_BETA         = 0.1   # KL penalty — keeps model near pretrained weightsTRAIN_TASKS     = ["basic_retrieval", "analytical_reasoning", "strategic_research"]# ── Step 1: Generate prompts from the environment ──────────────────────────print("  Generating training prompts...")prompts = []prompt_metadata = []for j in range(NUM_PROMPTS):    seed = j * 53 + 7    task = TRAIN_TASKS[j % len(TRAIN_TASKS)]    os.environ["FINQA_TASK"] = task    env = CostAwareFinqaEnvironment()    random.seed(seed)    env._question_index = seed % max(len(env._task_questions) if env._task_questions else 1, 1)    obs = env.reset()    messages = build_chat_messages(obs)    prompt_text = tokenizer.apply_chat_template(        messages, tokenize=False, add_generation_prompt=True,    )    prompts.append(prompt_text)    prompt_metadata.append({"seed": seed, "task": task, "question": obs.question[:80]})print(f"  Generated {len(prompts)} prompts across {len(TRAIN_TASKS)} tasks")# ── Step 2: Define the GRPO reward function ────────────────────────────────def grpo_reward_fn(completions, prompts_batch):    """Reward function for GRPO.    For each completion, we:    1. Parse the action from the generated text    2. Run it in the environment    3. Compute the shaped reward    """    rewards = []    for completion, prompt in zip(completions, prompts_batch):        # Find which prompt this corresponds to        prompt_idx = None        for idx, p in enumerate(prompts):            if p == prompt:                prompt_idx = idx                break        if prompt_idx is None:            rewards.append(0.0)            continue        meta = prompt_metadata[prompt_idx]        tool, query, answer = parse_action(completion)        # Run in environment        os.environ["FINQA_TASK"] = meta["task"]        score_env = CostAwareFinqaEnvironment()        random.seed(meta["seed"])        score_env._question_index = meta["seed"] % max(            len(score_env._task_questions) if score_env._task_questions else 1, 1        )        score_obs = score_env.reset()        act = CostAwareFinqaAction(tool=tool, query=query, answer=answer)        score_obs = score_env.step(act)        sr = shaped_reward(completion, {"tool": tool, "query": query, "answer": answer}, score_obs)        rewards.append(sr)    return rewards# ── Step 3: Prepare dataset for GRPO ──────────────────────────────────────train_data = Dataset.from_dict({"prompt": prompts})# ── Step 4: Train with GRPO ───────────────────────────────────────────────model = FastLanguageModel.for_training(model)grpo_config = GRPOConfig(    output_dir="./finqa_grpo",    per_device_train_batch_size=1,    gradient_accumulation_steps=4,    num_train_epochs=NUM_EPOCHS,    learning_rate=LEARNING_RATE,    warmup_steps=10,    fp16=not torch.cuda.is_bf16_supported(),    bf16=torch.cuda.is_bf16_supported(),    logging_steps=5,    max_completion_length=128,    num_generations=GROUP_SIZE,    beta=KL_BETA,             # KL penalty to prevent overtraining    save_strategy="no",       # Don't save checkpoints (Colab space)    report_to="none",)trainer = GRPOTrainer(    model=model,    args=grpo_config,    train_dataset=train_data,    reward_funcs=grpo_reward_fn,    processing_class=tokenizer,)print("  Starting GRPO training...")result = trainer.train()print(f"  GRPO training complete! Loss: {result.training_loss:.4f}")print(f"  Total steps: {result.global_step}")

## Cell 11 — LLM-as-Judge EvaluationUses a separate LLM (via HF Inference API) to evaluate the **process quality** ofthe trained agent — not just answer correctness, but whether the agent chosethe most cost-effective path.

In [ ]:
import urllib.requestdef llm_judge(question, tool_trace, final_answer, gold_answer, hf_token=None):    """Use an LLM to evaluate agent process quality.    Returns a score 0-1 and explanation.    """    token = hf_token or os.environ.get("HF_TOKEN", "")    if not token:        return 0.5, "No HF_TOKEN — skipping LLM judge"    judge_prompt = f"""You are evaluating a financial research agent's tool selection strategy.Question: {question}Gold answer: {gold_answer}Agent's answer: {final_answer}Agent's tool trace:{tool_trace}Rate the agent's PROCESS (not just the answer) on a scale of 0-1:- Did it use the cheapest sufficient tool?- Did it avoid unnecessary expensive calls?- Did it write good SQL queries (if applicable)?- Did it minimize total cost?Respond with JSON: {{"score": <0-1>, "explanation": "<brief reason>"}}"""    api_url = "https://router.huggingface.co/v1/chat/completions"    model = "Qwen/Qwen2.5-72B-Instruct"    payload = json.dumps({        "model": model,        "messages": [{"role": "user", "content": judge_prompt}],        "temperature": 0.1,        "max_tokens": 200,    })    try:        req = urllib.request.Request(            api_url,            data=payload.encode("utf-8"),            headers={                "Authorization": f"Bearer {token}",                "Content-Type": "application/json",            },        )        with urllib.request.urlopen(req, timeout=30) as resp:            data = json.loads(resp.read().decode())        response = data["choices"][0]["message"]["content"]        # Parse score from response        if "{" in response:            start = response.index("{")            end = response.rindex("}") + 1            result = json.loads(response[start:end])            return float(result.get("score", 0.5)), result.get("explanation", "")    except Exception as e:        return 0.5, f"Judge error: {e}"    return 0.5, "Could not parse judge response"# Test the judge with a sampleprint("Testing LLM-as-Judge...")test_score, test_explanation = llm_judge(    question="What was Intel's available-for-sale investments as % of total?",    tool_trace="Step 1: sql_query($0.001) -> Got financial table data\nStep 2: submit_answer(FREE) -> 0.532",    final_answer="0.532",    gold_answer="0.53232",)print(f"  Judge score: {test_score:.2f}")print(f"  Explanation: {test_explanation}")

## Cell 12 — Phase 4: Post-Training Evaluation

In [ ]:
print("\n" + "=" * 60)print("Phase 4 — Post-Training Evaluation")print("=" * 60)model = FastLanguageModel.for_inference(model)post_scores = []post_shaped = []post_costs  = []post_steps  = []judge_scores = []hf_token = os.environ.get("HF_TOKEN", "")for i, (seed, task) in enumerate(zip(SEEDS, EVAL_TASKS)):    res = run_episode(model, tokenizer, task=task, seed=seed,                      verbose=(i == 0))    post_scores.append(res["final_score"])    post_shaped.append(res["total_shaped"])    post_costs.append(res["cost_spent"])    post_steps.append(res["steps"])    # LLM-as-Judge evaluation (for first 4 episodes to conserve API calls)    if i < 4 and hf_token:        tool_trace = "\n".join([            f"Step {t['step']}: {t['action']['tool']}(${t['tool_cost']:.2f}) -> {t['gen_text'][:80]}"            for t in res["trajectory"]        ])        j_score, j_expl = llm_judge(            question=res["question"],            tool_trace=tool_trace,            final_answer=str(res["trajectory"][-1]["action"].get("answer", "")),            gold_answer="N/A",            hf_token=hf_token,        )        judge_scores.append(j_score)        print(f"  Ep {i+1:02d} [{task[:8]}]: score={res['final_score']:.4f}  "              f"cost=${res['cost_spent']:.2f}  judge={j_score:.2f}  "              f"tools={res['tools']}")    else:        print(f"  Ep {i+1:02d} [{task[:8]}]: score={res['final_score']:.4f}  "              f"shaped={res['total_shaped']:+.4f}  "              f"cost=${res['cost_spent']:.2f}  "              f"steps={res['steps']}  tools={res['tools']}")print(f"\n  >> Post-train score mean:  {np.mean(post_scores):.4f} +/- {np.std(post_scores):.4f}")print(f"  >> Post-train shaped mean: {np.mean(post_shaped):.4f} +/- {np.std(post_shaped):.4f}")print(f"  >> Post-train avg cost:    ${np.mean(post_costs):.2f}")print(f"  >> Score improvement:      {np.mean(post_scores) - np.mean(baseline_scores):+.4f}")print(f"  >> Cost reduction:         ${np.mean(baseline_costs) - np.mean(post_costs):+.2f}")if judge_scores:    print(f"  >> LLM Judge avg score:    {np.mean(judge_scores):.2f}")

## Cell 13 — Comparison Plots

In [ ]:
import matplotlib.pyplot as pltimport matplotlibmatplotlib.rcParams.update({"font.size": 12})fig, axes = plt.subplots(1, 3, figsize=(18, 5))# 1. Bar chart: Score comparisonlabels = ["Random\nAgent", "Pre-Train\nLLM", "GRPO-Trained\nLLM"]means  = [np.mean(random_scores), np.mean(baseline_scores), np.mean(post_scores)]stds   = [np.std(random_scores),  np.std(baseline_scores),  np.std(post_scores)]colors = ["#95a5a6", "#e74c3c", "#2ecc71"]bars = axes[0].bar(labels, means, yerr=stds, color=colors, capsize=8,                   edgecolor="black", linewidth=0.8)axes[0].set_ylabel("Mean Score")axes[0].set_title("Cost-Aware FinQA — Agent Comparison")axes[0].grid(axis="y", alpha=0.3)if means[2] > means[1]:    delta = means[2] - means[1]    pct = (delta / max(abs(means[1]), 0.001)) * 100    axes[0].annotate(f"+{pct:.0f}%\n({delta:+.3f})",                     xy=(2, means[2] + stds[2] + 0.02),                     fontsize=11, fontweight="bold", color="#27ae60",                     ha="center", va="bottom")# 2. Per-episode comparisonx = range(1, NUM_EVAL + 1)axes[1].plot(x, baseline_scores, "o-", color="#e74c3c", linewidth=2,             markersize=7, label="Pre-train")axes[1].plot(x, post_scores, "s-", color="#2ecc71", linewidth=2,             markersize=7, label="GRPO-trained")axes[1].fill_between(x, baseline_scores, post_scores,                     alpha=0.15, color="#2ecc71",                     where=[p > b for p, b in zip(post_scores, baseline_scores)])axes[1].set_xlabel("Episode")axes[1].set_ylabel("Final Score")axes[1].set_title("Per-Episode Score")axes[1].legend(loc="lower right")axes[1].grid(alpha=0.3)# 3. Cost comparisoncost_labels = ["Random", "Pre-Train", "GRPO"]cost_means = [np.mean(random_costs), np.mean(baseline_costs), np.mean(post_costs)]cost_colors = ["#95a5a6", "#e74c3c", "#2ecc71"]axes[2].bar(cost_labels, cost_means, color=cost_colors,            edgecolor="black", linewidth=0.8)axes[2].set_ylabel("Mean Cost ($)")axes[2].set_title("Average Episode Cost")axes[2].grid(axis="y", alpha=0.3)for bar, val in zip(axes[2].patches, cost_means):    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,                f"${val:.2f}", ha="center", fontweight="bold")plt.tight_layout()plt.savefig("finqa_training_results.png", dpi=150, bbox_inches="tight")plt.show()print("Plot saved to finqa_training_results.png")

## Cell 14 — Detailed Episode Walkthrough

In [ ]:
print("\n" + "=" * 60)print("Detailed Episode Walkthrough (GRPO-trained)")print("=" * 60)for task in ["basic_retrieval", "analytical_reasoning", "strategic_research"]:    print(f"\n--- Task: {task} ---")    res = run_episode(model, tokenizer, task=task, seed=42, verbose=True)    print(f"  Final score: {res['final_score']:.4f}")    print(f"  Cost spent: ${res['cost_spent']:.2f} / ${res['budget_total']:.2f}")    print(f"  Tools used: {res['tools']}")    print(f"  Question: {res['question']}")

## Cell 15 — Final Summary

In [ ]:
print("\n" + "=" * 70)print("  COST-AWARE FINQA — GRPO TRAINING RESULTS")print("=" * 70)print(f"  Environment:   Cost-Aware FinQA")print(f"  HF Space:      {SPACE_URL}")print(f"  Model:         {MODEL_NAME}")print(f"  LoRA rank:     {LORA_R}")print(f"  Training:      GRPO (group_size={GROUP_SIZE}, KL_beta={KL_BETA})")print(f"                 {NUM_PROMPTS} prompts, {NUM_EPOCHS} epochs")print(f"  " + "-" * 50)print(f"  {'Metric':<22s} {'Score':>10s}  {'Shaped':>10s}  {'Avg Cost':>10s}")print(f"  {'-'*22} {'-'*10}  {'-'*10}  {'-'*10}")print(f"  {'Random agent':<22s} {np.mean(random_scores):>+10.4f}  {'N/A':>10s}  ${np.mean(random_costs):>8.2f}")print(f"  {'Pre-train LLM':<22s} {np.mean(baseline_scores):>+10.4f}  {np.mean(baseline_shaped):>+10.4f}  ${np.mean(baseline_costs):>8.2f}")print(f"  {'GRPO-trained LLM':<22s} {np.mean(post_scores):>+10.4f}  {np.mean(post_shaped):>+10.4f}  ${np.mean(post_costs):>8.2f}")score_imp = np.mean(post_scores) - np.mean(baseline_scores)score_pct = (score_imp / max(abs(np.mean(baseline_scores)), 0.001)) * 100print(f"  " + "-" * 50)print(f"  Score delta:    {score_imp:+.4f}  ({score_pct:+.0f}%)")if judge_scores:    print(f"  LLM Judge avg:  {np.mean(judge_scores):.2f}")print("=" * 70)print()print("Key insight: GRPO trains the agent to prefer cheap SQL queries over")print("expensive tools, reducing cost while maintaining accuracy — without")print("overtraining or losing general-purpose ability (low LoRA rank + KL penalty).")

---## Next Steps1. **Scale up**: Increase `NUM_PROMPTS` and `NUM_EPOCHS` for better results2. **Multi-step GRPO**: Train on full episode trajectories instead of single steps3. **Larger models**: Try Qwen 2.5-7B or 14B for more capacity4. **Custom database**: Replace the sample FinQA data with your own financial database   - Update `/data/financial_data.db` with your data   - Modify `/server/tools.py` to match your schema   - See DESIGN.md for full instructions